# Gyroids_STL

**What this notebook showcases in `triply`:** the core, function-based TPMS/gyroid generation workflow. It builds a 3D design-space grid, computes a gyroid scalar field over that grid (`gyroid.py`), extracts an isosurface mesh with marching cubes (`mesh_tools.py`), and then previews/exports the result as an STL surface mesh (`viz.py`, `io_ops.py`). This is the procedural (non-class) API for creating gyroid structures.


In [1]:
import numpy as np
from stl import mesh as stl_mesh
import plotly.graph_objects as go  # for visualization
import os
import trimesh


import triply
from triply.utils import reload_all
reload_all()

working_path = os.getcwd()
print("Current working directory:", working_path)




[gyroid_utils] version 3.1.4 loaded
[gyroid_utils] version 3.1.4 loaded
gyroid_utils: all modules reloaded
Current working directory: c:\Users\cofo\Documents\02 - GitHub\GYROIDS\notebooks


## 1 — Define the Design Space

Set up the 3D coordinate grid that the gyroid scalar field will be evaluated on.

- **Domain size**: 100 × 100 × 100 units along *x*, *y*, and *z*.
- **Grid spacing** (`dx_grid`, `dy_grid`, `dz_grid`): computed as `domain_size / 25`, giving 26 nodes per axis (including the endpoint).

The min/max of each axis and the total number of voxels are printed as a sanity check before proceeding.


In [2]:
# size of domain
pz2 = 100
py2 = 100
px2 = 100

# --- Discretization of the domain ---
# Resolution in each axis, calculated as a funcion of the size of the gyroid's unit cell
dx_grid = px2 / 25
dy_grid = py2 / 25
dz_grid = pz2 / 25

# 1D coordinate arrays. np.arange(stop + step, step) includes the endpoint like MATLAB's colon with step.
x1 = np.arange(0, px2 + dx_grid, dx_grid)       # x positions from 0 to pz2 + dx_grid, and the step size is dx_grid
y1 = np.arange(0, py2 + dy_grid, dy_grid)       # y positions from 0 to py2, step dy_grid
z1 = np.arange(0, pz2 + dz_grid, dz_grid)       # z positions from 0 to pz2, step dz_grid

# Create 3D coordinate grids. indexing='ij' -> (X,Y,Z) follow x1,y1,z1 order like MATLAB.
x, y, z = np.meshgrid(x1, y1, z1, indexing='ij')
print(np.min(x),np.max(x))
print(np.min(y),np.max(y))
print(np.min(z),np.max(z))

print(f"x-axis resolution {np.size(x1)=}, y-axis resolution {np.size(y1)=}, z-axis resolution {np.size(z1)=}")
print(f"In total, {np.size(x)} voxels in the 3D grid")

0.0 100.0
0.0 100.0
0.0 100.0
x-axis resolution np.size(x1)=26, y-axis resolution np.size(y1)=26, z-axis resolution np.size(z1)=26
In total, 17576 voxels in the 3D grid


## 2 — Define the Gyroid Field (manual formula)

Unlike `Gyroids_STL_class.ipynb`, this notebook computes the gyroid scalar field directly with the raw trigonometric formula instead of going through the `GyroidModel` class, showing what happens "under the hood" of `compute_field()`.

1. **Set the unit-cell period** — `px`, `py`, `pz`, here constant over the whole domain.
2. **Set the thickness offset `t`** — controls wall thickness/offset from the mid-surface (here a constant `0.5`).
3. **Compute the scalar field `v`** — absolute value of the gyroid trigonometric function minus `t`, then negated (equivalent to the `"abs"` mode of `GyroidModel.compute_field()`).
4. **`twod_view_of_matrix`** — renders a 2-D slice of the field for a visual sanity-check.


In [ ]:
# --------- Create gyroids -----------
file_name = "gyroid-test"

# ---  Define Period of gyroid unit cell -------
px = np.zeros_like(x) + 100
py = np.zeros_like(y) + 100
pz = np.zeros_like(z) + 100


# ------- check for errors -------
#in the period of the gyroid is too small for the grid resolution, it will cause errors in the marching cubes algorithm
if np.min(pz) <= 5 * dz_grid :
    print(f"max pz: {np.max(pz)}, min pz: {np.min(pz)}, resultion dz_grid: {dz_grid}")
    raise ValueError("Period too small for the grid resolution. z-axis.")   
elif np.min(py) <= 5 * dy_grid :
    print(f"max pz: {np.max(py)}, min pz: {np.min(py)}, resultion dz_grid: {dy_grid}")
    raise ValueError("Period too small for the grid resolution. y-axis.")   
elif np.min(px) <= 5 * dx_grid:
    print(f"max pz: {np.max(px)}, min pz: {np.min(px)}, resultion dz_grid: {dx_grid}")
    raise ValueError("Period too small for the grid resolution. z-axis.")

# ------ Define Thickness function t ---   #can be from -1.4265847744427516 + 1.4265847744427516
t = np.zeros_like(x) + 0.5


# --- Gyroid scalar field v (isosurface at v=0 gives the gyroid surface) ---
v = np.abs( np.sin((2*np.pi/px)*x) * np.cos((2*np.pi/py)*y)
    + np.sin((2*np.pi/py)*y) * np.cos((2*np.pi/pz)*z)
    + np.sin((2*np.pi/pz)*z) * np.cos((2*np.pi/px)*x) ) - t # add thickness field
v=-v

# --- Visualize result ---

triply.viz.twod_view_of_matrix(v, x1, y1, z1, 0, 0.01)

## 3 — Generate the Surface Mesh (STL)

Turns the scalar field `v` into a cleaned-up, exportable surface mesh.

- **`mesh_from_matrix`:** extracts the iso-surface via Marching Cubes. It uses boundary padding so the mesh closes at the domain edge.
- **`simplify_mesh`:** reduces the triangle count down to a `target` of 10,000 faces.
- **`keep_largest_connected_component`:** discards any small disconnected fragments.
- **`save_mesh_as_html`** / **`export_as_STL`:**  writes an interactive 3-D preview and the final `.stl` file.
- **`calculate_triangle_areas`** + **`plot_histogram`:** visualizes the triangle-area distribution as a mesh-quality check.
- **`check_mesh_validity`:** reports watertightness, manifoldness, and self-intersections.


In [ ]:
verts, faces = triply.mesh_tools.mesh_from_matrix(
        matrix=v,
        iso_level=0,
        algo_step_size=3,
        x = x,
        y = y,
        z = z,
        pad_width = 5,
    )

print(f"there are {len(faces)} faces in this model")
# simplify and clean the mesh
verts, faces = triply.mesh_tools.simplify_mesh(verts, faces, target=10000)
verts, faces = triply.mesh_tools.keep_largest_connected_component(verts, faces)

# ---- save the mesh ----
triply.viz.save_mesh_as_html(faces, verts, file_name)
#export_as_STL_2(verts, faces, f"{working_path}/{file_name}.stl")
triply.mesh_tools.export_as_STL(verts, faces, f"{file_name}.stl")

# ------ visualize the quality of your mesh ----
triangle_areas = triply.mesh_tools.calculate_triangle_areas(verts, faces)
triply.viz.plot_histogram(triangle_areas)
triply.mesh_tools.check_mesh_validity(verts,faces)

## 4 — Reload and Inspect the Result

Round-trips the exported STL and inspects the underlying fields.

- **`load_stl`** + **`check_mesh_validity`** — reloads the `.stl` file just written and re-confirms it is watertight/manifold.
- **Plot of `v` along the x-axis** — shows the raw scalar field profile used to generate the mesh.
- **Plot of the thickness function `t`** along *x*, *y*, and *z* — mainly useful when `t` is a spatially-varying array rather than a single constant.


In [6]:
verts, faces = triply.io_ops.load_stl(working_path + '/' + file_name + '.stl')
triply.mesh_tools.check_mesh_validity(verts,faces)


[INFO] gyroid_utils:((load_stl)): Loading STL file: c:\Users\cofo\Documents\02 - GitHub\GYROIDS\notebooks/gyroid-test.stl
[INFO] gyroid_utils:((load_stl)): Loaded STL successfully: 840 vertices, 1716 faces
[INFO] gyroid_utils:((check_mesh_validity)): check_mesh_validity(): {'watertight': True, 'winding_consistent': True, 'is_volume': True, 'boundary_edges': 0, 'nonmanifold_edges': 0, 'self_intersecting': False}


{'watertight': True,
 'winding_consistent': True,
 'is_volume': True,
 'boundary_edges': 0,
 'nonmanifold_edges': 0,
 'self_intersecting': False}

In [ ]:
fig_2 = go.Figure()
fig_2.add_trace(go.Scatter(
    x=x[:,0,0],
    y=v[:,0,0],
    mode='lines+markers',
    name='Example Line'
))
fig_2.update_layout(title="v function along x-axis",
                    template="plotly_white")
fig_2.show()

In [ ]:
fig_0 = go.Figure()
fig_0.add_trace(go.Scatter(
    x=x[:,0,0],
    y=t[:,0,0],
    mode='lines+markers',
    name='x axis'
))
fig_0.add_trace(go.Scatter(
    x=y[0,:,0],
    y=t[0,:,0],
    mode='lines+markers',
    name='y axis'
))
fig_0.add_trace(go.Scatter(
    x=z[0,0,:],
    y=t[0,0,:],
    mode='lines+markers',
    name='z axis'
))
fig_0.update_layout(title="Thickness function along x,y and z-axis",
                    template="plotly_white")
fig_0.show()

